# Baseline Training with Pipeline

Builds a scikit learn pipeline to run the baseline training

In [6]:
# load data

import polars as pl
import pandas as pd
from pathlib import Path

PROCESSED_FOLDER = Path("./../data/processed/")

data = pl.read_parquet(PROCESSED_FOLDER / "train.parquet").to_pandas()
data.head()

,date,store_nbr,item_nbr,unit_sales,onpromotion
0,2013-01-01,25,103665,7.0,None
1,2013-01-01,25,105574,1.0,None
2,2013-01-01,25,105575,2.0,None
3,2013-01-01,25,108079,1.0,None
4,2013-01-01,25,108701,1.0,None


In [7]:
# preprocess data
data["unit_sales"] = data["unit_sales"].clip(lower=0)

# adds 0 sales for missing item sales
sales = data.groupby(["date", "store_nbr", "item_nbr"])["unit_sales"].sum()

all_dates = sales.index.get_level_values("date").unique()
all_items = sales.index.get_level_values("item_nbr").unique()
all_shops = sales.index.get_level_values("store_nbr").unique()

full_index = pd.MultiIndex.from_product([all_dates, all_shops, all_items], names=["date", "store_nbr", "item_nbr"])

data_zeros = sales.reindex(full_index, fill_value=0).reset_index()

data_merged = data_zeros.merge(data.drop(columns="unit_sales"), on=["date", "item_nbr", "store_nbr"], how="left")
data_merged.head()

,date,store_nbr,item_nbr,unit_sales,onpromotion
0,2013-01-01,25,103665,7.0,None
1,2013-01-01,25,105574,1.0,None
2,2013-01-01,25,105575,2.0,None
3,2013-01-01,25,108079,1.0,None
4,2013-01-01,25,108701,1.0,None


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import TargetEncoder, FunctionTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
import numpy as np

df = data_merged

def clean_features(d):
    d = d.copy()
    d["onpromotion"] = d["onpromotion"].fillna(False)
    return d

def add_date_features(d):
    d = d.copy()
    # adds weekday and month feature
    d["date"] = pd.to_datetime(d["date"])
    d["weekday_num"] = d["date"].dt.day_of_week
    d["month_num"] = d["date"].dt.month
    return d.drop(columns="date")

train = df[(df["date"] < "2017-07-31") & (df["date"] > "2017-07-15")].sample(n=2_000_000, random_state=0)
valid = df[df["date"] >= "2017-07-31"].sample(n=400_000, random_state=0)

# Use the raw item_nbr and the raw unit_sales; the pipeline handles both
features = ["date", "store_nbr", "item_nbr", "onpromotion"]
X_train, y_train = train[features], train["unit_sales"]
X_valid, y_valid = valid[features], valid["unit_sales"]

# Target-encode item_nbr and pass the other columns through unchanged
preprocess = ColumnTransformer(
    [("item_te", TargetEncoder(), ["item_nbr"])],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")   # keeps store_nbr as 'category'

model = TransformedTargetRegressor(
    regressor=make_pipeline(
        FunctionTransformer(clean_features),
        FunctionTransformer(add_date_features),
        preprocess,
        HistGradientBoostingRegressor(
            categorical_features=["store_nbr"],
            max_iter=1000, learning_rate=0.05, max_leaf_nodes=127,
            early_stopping=True,
        ),
    ),
    func=np.log1p,
    inverse_func=np.expm1,
    check_inverse=False,   # clipping can't be undone, so skip the round-trip check
)


model.fit(X_train, y_train)
pred = model.predict(X_valid)

In [24]:
# print error
from sklearn.metrics import root_mean_squared_log_error

error = root_mean_squared_log_error(pred, valid["unit_sales"])
print(f"Error for random sample set is: {error}")

Error for random sample set is: 0.759650887339876


# Results:

* add 0 sales -> add weekday and month -> HistGradientBoostingRegressor : Error = 0.9040307131801688
* add 0 sales -> add weekday and month -> only use july for training and last 16 days for testing -> HistGradientBoostingRegressor : Error = 0.7605423352457179
* add 0 sales -> add weekday and month -> only use from july 15th for training and last 16 days for testing -> HistGradientBoostingRegressor : Error = 0.759650887339876

In [ ]:
# create csv
import os

RESULT_FOLDER = Path("./../data/results") 
EXTRACTED_FOLDER = Path("./../data/raw") / "extracted"

# train model
train = df[df["date"] >= "2017-07-31"].sample(n=2_000_000, random_state=0)
X_train, y_train = train[features], train["unit_sales"]
model.fit(X_train, y_train)

# make predictions
test_data = pd.read_csv(EXTRACTED_FOLDER / "test.csv")
test_pred = np.expm1(model.predict(test_data[features]))

# write to csv
submission_data = pd.DataFrame({
    "id": test_data["id"].to_numpy(),
    "unit_sales": np.asarray(test_pred, dtype=float).ravel(),
})

os.makedirs(RESULT_FOLDER, exist_ok=True)
submission_data.to_csv(RESULT_FOLDER / "baseline_result_v2.csv", index=False)

KeyError: 'id'